In [39]:
import pandas as pd
import numpy as np

cv_data = pd.read_csv('cv_cleaned.csv')
cv_data.set_index('Urban Center', inplace=True)

x_col = ['Population Density', 'Rapid Transit to Resident Ratio (RTR)', 
        'Oxford Economics Global Cities Index (Quality of Life)',
        'Percentage Access to Public Transport', 'Average travel speed (2019)', 
        'Built-up areas per capita']

cleaned = cv_data[x_col].copy()
cleaned['Percentage Access to Public Transport'] *= 100

# Zero imputation for null RTR (usually in smaller cities with no formal rapid transit)
cleaned['Rapid Transit to Resident Ratio (RTR)'] = cleaned['Rapid Transit to Resident Ratio (RTR)'].fillna(0)
cleaned = cleaned.dropna()

normalised = pd.DataFrame(columns=x_col)
for col in x_col:
    normalised[col] = (cleaned[col]-cleaned[col].mean()) / cleaned[col].std()

normalised['Percentage Access to Public Transport'] = cv_data['Percentage Access to Public Transport']

In [40]:
# 1. Run OLS and check model performance for multiple variables
import statsmodels.api as sm

# Access ~ density + rtr + qol + travel speed + built up area pc
x_col.remove('Percentage Access to Public Transport')

X = sm.add_constant(normalised[x_col])
y = normalised['Percentage Access to Public Transport'].values
model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.593
Model:                            OLS   Adj. R-squared:                  0.576
Method:                 Least Squares   F-statistic:                     35.29
Date:                Tue, 12 May 2026   Prob (F-statistic):           3.93e-22
Time:                        01:25:11   Log-Likelihood:                 223.07
No. Observations:                 127   AIC:                            -434.1
Df Residuals:                     121   BIC:                            -417.1
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                                             coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------

In [41]:
# Access ~ density + rtr + qol + travel speed
x_col.remove('Built-up areas per capita')

X = sm.add_constant(normalised[x_col])
y = normalised['Percentage Access to Public Transport'].values
model = sm.OLS(y, X).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.584
Model:                            OLS   Adj. R-squared:                  0.570
Method:                 Least Squares   F-statistic:                     42.77
Date:                Tue, 12 May 2026   Prob (F-statistic):           2.22e-22
Time:                        01:25:11   Log-Likelihood:                 221.60
No. Observations:                 127   AIC:                            -433.2
Df Residuals:                     122   BIC:                            -419.0
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                                             coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------

In [42]:
# Access ~ density + rtr + qol
x_col.remove('Average travel speed (2019)')

X = sm.add_constant(normalised[x_col])
y = normalised['Percentage Access to Public Transport'].values
model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.581
Model:                            OLS   Adj. R-squared:                  0.571
Method:                 Least Squares   F-statistic:                     56.79
Date:                Tue, 12 May 2026   Prob (F-statistic):           4.13e-23
Time:                        01:25:11   Log-Likelihood:                 221.15
No. Observations:                 127   AIC:                            -434.3
Df Residuals:                     123   BIC:                            -422.9
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                                             coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------

In [ ]:
# Access ~ rtr + qol
x_col.remove('Population Density')

X = sm.add_constant(normalised[x_col])
y = normalised['Percentage Access to Public Transport'].values
model = sm.OLS(y, X).fit()
print(model.summary())

# Parsimonious model identified with 2 independent variables
# Only RTR and QoL are strong and significant predictors of public transport access
# R^2 from 5 variables to 2 (55.7%, 52.2%, 52.1%, 45%) in EDA 
# R^2 from 5 variables to 2 (59.3%, 58.4%, 58.1%, 58%) in cross validation

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.580
Model:                            OLS   Adj. R-squared:                  0.573
Method:                 Least Squares   F-statistic:                     85.51
Date:                Tue, 12 May 2026   Prob (F-statistic):           4.59e-24
Time:                        01:25:11   Log-Likelihood:                 220.99
No. Observations:                 127   AIC:                            -436.0
Df Residuals:                     124   BIC:                            -427.4
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                                                             coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------

In [ ]:
# 2. Run access ~ rtr + qol (parsimonious model) on full dataset

eda_data = pd.read_csv('cleaned.csv')
eda_data.set_index('Urban Center', inplace=True)
full_data = pd.concat([eda_data, cv_data])
full_data.drop('London')

# Zero imputation for null RTR (usually in smaller cities with no formal rapid transit)
x_col = ['Rapid Transit to Resident Ratio (RTR)', 
         'Oxford Economics Global Cities Index (Quality of Life)', 'Percentage Access to Public Transport']
no_rtr = full_data[full_data['Rapid Transit to Resident Ratio (RTR)'].isna()] 
print("Cities with unknown / zero formal rapid transit: ", no_rtr.index.tolist())
full_data['Rapid Transit to Resident Ratio (RTR)'] = full_data['Rapid Transit to Resident Ratio (RTR)'].fillna(0)

full_cleaned = full_data[x_col]
full_cleaned = full_cleaned.dropna()
full_cleaned['Percentage Access to Public Transport'] *= 100

# Normalise columns
full_normalised = pd.DataFrame(columns=x_col)
for col in x_col:
    full_normalised[col] = (full_cleaned[col]-full_cleaned[col].mean())/full_cleaned[col].std()

full_normalised['Percentage Access to Public Transport'] = full_cleaned['Percentage Access to Public Transport'].copy()

Cities with unknown / zero formal rapid transit:  ['Kabul', 'Melbourne', 'Chattogram', 'Dhaka', 'Ho Chi Minh City', 'Baghdad', 'Lagos', 'Riyadh', 'Chandigarh', 'Yangon', 'Colombo', 'Phnom Penh', 'Patna', 'Bandung', 'Surabaya', 'Bishkek', 'Vientiane', 'Ulaanbaatar', 'Mandalay', 'Kathmandu', 'Karachi', 'Cebu City', 'Davao City', 'Ulsan', 'Kazan', 'Nizhny Novgorod', 'Taichung', 'Luanda', 'Abidjan', 'Kinshasa', 'Port-au-Prince', 'Nairobi', 'Khartoum', 'Detroit', 'San Jose', 'Thimphu', 'Avarua', 'South Tarawa', 'Alofi', 'Apia', "Nuku'alofa", 'Funafuti', 'Port Vila', 'Herat', 'Kandahar', 'Perth', 'Bogura', 'Brahmanbaria', 'Comilla', 'Daganbhuiyan', 'Jessore', 'Khulna', 'Mymensingh', 'Rajshahi', 'Rangpur', 'Sylhet', 'Tangail', 'Bandar Seri Begawan', 'Anshan', 'Anyang', 'Baoding', 'Baotou', 'Bengbu', 'Changshu', 'Dandong', 'Fancheng', 'Fushun', 'Guilin', 'Haikou', 'Handan', 'Hengyang', 'Huaibei', 'Huangshi', 'Huizhou', 'Jieyang', 'Jilin', 'Jingzhou', 'Jinzhou', 'Linyi', 'Liuzhou', 'Mianyang', 

In [45]:
x_col.remove('Percentage Access to Public Transport')

X = sm.add_constant(full_normalised[x_col])
y = full_normalised['Percentage Access to Public Transport'].values
model = sm.OLS(y, X).fit()
print(model.summary())

# Model is stable on full dataset. F statistic is high. R squared (adjusted + not adjusted) values are constant and explains 65% of variation
# PT access appears primarily driven by rapid transit infrastructure porovision + social development (quality of life) 

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.646
Model:                            OLS   Adj. R-squared:                  0.644
Method:                 Least Squares   F-statistic:                     306.8
Date:                Tue, 12 May 2026   Prob (F-statistic):           1.57e-76
Time:                        01:25:11   Log-Likelihood:                -1177.1
No. Observations:                 339   AIC:                             2360.
Df Residuals:                     336   BIC:                             2372.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                                                             coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------

In [47]:
pred = pd.Series(model.predict(X), index=full_normalised.index)

# Force DataFrame to Series 
full_normalised['residual'] = full_normalised['Percentage Access to Public Transport'].squeeze()-pred

above = full_normalised[full_normalised['residual']>=0]
below = full_normalised[full_normalised['residual']<0]

a = above[above.index.isin(no_rtr.index.tolist())]
print("Above no RTR: ", a.shape[0])

b = below[below.index.isin(no_rtr.index.tolist())]
print("Below no RTR: ", b.shape[0])

print("0 RTR 0 Access Cities: ", a[a['Percentage Access to Public Transport']==0].shape[0]+b[b['Percentage Access to Public Transport']==0].shape[0])

print("0 RTR non-zero Access Cities: ", a[a['Percentage Access to Public Transport']!=0].index.tolist()+b[b['Percentage Access to Public Transport']!=0].index.tolist())
print("n=12")

# 12 cities without rapid transit infrastructure have non zero access. Some of them may possess non-grade separated public transport infrastructure (trams, bus network, informal sector)  
# Next step: ideation for data product (1. benchmarking cities: what if a city invest 20% more in rapid transit?, show me cities similar to me that have x2 access; 2. city archetype using spider chart; 3. transit efficiency score)  

Above no RTR:  51
Below no RTR:  148
0 RTR 0 Access Cities:  187
0 RTR non-zero Access Cities:  ['Dhaka', 'Lagos', 'Kazan', 'Taichung', 'San Jose', 'Perth', 'Huizhou', 'Nantong', 'Melbourne', 'Detroit', 'Yiwu', 'Yueqing']
n=12
